# 02 - Reading Pulses And Making First Plots

In this notebook we turn Physics frames into ordinary Python rows. This is the bridge between IceTray objects and familiar analysis tools like Pandas and Matplotlib.

The code is written out directly so you can see the small steps: find a pulse map, count hit DOMs, add up charge, and save one row per event.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from icecube import dataio, dataclasses

SIM_FILE = Path('/data/sim/IceCube/2020/filtered/level2/CORSIKA-in-ice/20904/0000000-0000999/Level2_IC86.2020_corsika.020904.000000.i3.zst')
DATA_FILE = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_Subrun00000000_00000000.i3.zst')

print('Experimental data file exists:', DATA_FILE.exists())
print('Simulation file exists:       ', SIM_FILE.exists())


## Small helper functions

These helpers are intentionally simple. They are not meant to be a perfect library; they are meant to show the logic clearly.

A pulse map is usually stored under a key such as `SplitInIcePulses`. The exact key can change between files, so we try a short list of common possibilities.


In [ ]:
def frame_stop(frame):
    names = {'Q': 'DAQ', 'P': 'Physics', 'G': 'Geometry', 'C': 'Calibration', 'D': 'DetectorStatus', 'I': 'TrayInfo'}
    return names.get(str(frame.Stop), str(frame.Stop))

def event_header_as_dict(frame):
    if 'I3EventHeader' not in frame:
        return {}
    header = frame['I3EventHeader']
    return {
        'run_id': header.run_id,
        'event_id': header.event_id,
        'sub_event_id': header.sub_event_id,
        'sub_event_stream': str(header.sub_event_stream),
    }

def find_pulse_key(frame):
    possible_keys = ['SplitInIcePulses', 'SplitInIceDSTPulses', 'SRTInIcePulses', 'InIcePulses', 'OfflinePulses']
    for key in possible_keys:
        if key in frame:
            return key
    return None

def summarize_pulses(frame):
    pulse_key = find_pulse_key(frame)
    if pulse_key is None:
        return None

    # from_frame works for regular pulse maps and for common pulse masks.
    pulse_map = dataclasses.I3RecoPulseSeriesMap.from_frame(frame, pulse_key)

    hit_doms = 0
    number_of_pulses = 0
    total_charge = 0.0

    for omkey, pulses_on_one_dom in pulse_map:
        hit_doms += 1
        number_of_pulses += len(pulses_on_one_dom)
        for pulse in pulses_on_one_dom:
            total_charge += pulse.charge

    return {
        'pulse_key': pulse_key,
        'hit_doms': hit_doms,
        'number_of_pulses': number_of_pulses,
        'total_charge': total_charge,
    }

print('Defined beginner-friendly helpers for frame stops, headers, and pulses.')


## Optional: get a simple reconstructed direction

Many Level2 files contain several reconstructed particles. A reconstructed particle often has a direction with zenith and azimuth angles. We try a few common keys and keep the first one that exists.


In [ ]:
def reconstruction_as_dict(frame):
    possible_reco_keys = ['SplineMPE', 'MPEFit', 'LineFit', 'OnlineL2_SplineMPE']
    for key in possible_reco_keys:
        if key in frame:
            particle = frame[key]
            return {
                'reco_key': key,
                'zenith_rad': float(particle.dir.zenith),
                'azimuth_rad': float(particle.dir.azimuth),
                'energy_GeV': float(getattr(particle, 'energy', float('nan'))),
            }
    return {'reco_key': None, 'zenith_rad': float('nan'), 'azimuth_rad': float('nan'), 'energy_GeV': float('nan')}

print('Defined a helper that looks for a few common reconstruction keys.')


## Build an event table

Now we loop over Physics frames. For each Physics frame, we create a normal Python dictionary. At the end, Pandas turns the list of dictionaries into a table.


In [ ]:
def build_event_table(i3_path, max_physics_frames=500):
    rows = []
    physics_frames_seen = 0
    frames_with_pulses = 0

    i3_file = dataio.I3File(str(i3_path))
    while i3_file.more() and physics_frames_seen < max_physics_frames:
        frame = i3_file.pop_frame()
        if frame_stop(frame) != 'Physics':
            continue

        physics_frames_seen += 1
        row = event_header_as_dict(frame)

        pulse_summary = summarize_pulses(frame)
        if pulse_summary is not None:
            frames_with_pulses += 1
            row.update(pulse_summary)

        row.update(reconstruction_as_dict(frame))
        rows.append(row)

    i3_file.close()

    print(f'Read {physics_frames_seen} Physics frames from {i3_path.name}.')
    print(f'Frames with a recognized pulse key: {frames_with_pulses}')
    return pd.DataFrame(rows)

data_events = build_event_table(DATA_FILE, max_physics_frames=500)
print('Columns in the table:', list(data_events.columns))
data_events.head()


## Plot hit DOMs and charge

A DOM counts as hit if it has at least one pulse in the pulse map. Total charge is a rough event-size variable: bigger events tend to have more charge.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

data_events['hit_doms'].dropna().hist(ax=axes[0], bins=40)
axes[0].set_xlabel('hit DOMs')
axes[0].set_ylabel('events')
axes[0].set_title('DOMs hit per event')

data_events['total_charge'].dropna().hist(ax=axes[1], bins=40)
axes[1].set_xlabel('total charge [PE]')
axes[1].set_ylabel('events')
axes[1].set_title('Total pulse charge')

plt.tight_layout()
print('These histograms came from ordinary Pandas columns built from I3 frames.')


## Compare experimental data to simulation

This is not a careful physics comparison yet. We are only checking that the same code can summarize both files.


In [ ]:
sim_events = build_event_table(SIM_FILE, max_physics_frames=500)

ax = data_events['hit_doms'].dropna().plot.hist(bins=40, alpha=0.5, label='experimental data')
sim_events['hit_doms'].dropna().plot.hist(ax=ax, bins=40, alpha=0.5, label='simulation')
ax.set_xlabel('hit DOMs')
ax.set_ylabel('events')
ax.legend()
print('The two samples may have different shapes because they are not normalized or selected in a careful way yet.')


## Practice

1. Change `max_physics_frames` from 500 to 1000 and rerun the table-building cell.
2. Add a new table column for zenith in degrees instead of radians.
3. Print the pulse key used for the first 10 events. Is it always the same?
4. Add another candidate pulse key to `find_pulse_key` and see whether it changes anything.
